# 06 - Poisson Equation

In this notebook, we solve the Poisson equation using PINNs:

$$-\nabla^2 u = f(x)$$

This is a steady-state equation (no time dependence) used in electrostatics, gravity, and heat conduction.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import matplotlib.pyplot as plt
from pinns import Config, MLP, Trainer, set_seed
from pinns.equations.poisson import PoissonEquation1D
from pinns.evaluation import plot_solution_1d, compute_all_metrics

In [ ]:
set_seed(42)

config = Config(
    name='poisson_1d',
    equation_name='poisson_1d',
    equation_params={'source_fn': lambda x: -torch.sin(torch.pi * x)},
    model={'type': 'mlp', 'hidden_layers': [32, 32, 32], 'activation': 'tanh'},
    training={'n_collocation': 500, 'n_boundary': 100, 'epochs': 3000, 'lr': 1e-3},
)

eq = PoissonEquation1D(source_fn=lambda x: -torch.sin(torch.pi * x))
print(f"Domain: {eq.domain}")

In [ ]:
model = MLP(
    input_dim=1,  # x only
    output_dim=1,
    hidden_layers=[32, 32, 32],
    activation='tanh'
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
trainer = Trainer(
    model=model,
    equation=eq,
    config=config,
    print_every=500
)

history = trainer.train()

plt.figure(figsize=(10, 4))
plt.plot(history['total_loss'], label='Total Loss')
plt.plot(history['physics_loss'], label='Physics Loss')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.yscale('log')
plt.legend()
plt.title('Poisson Equation Training')
plt.show()

In [ ]:
import numpy as np

x = torch.linspace(0, 1, 100).reshape(-1, 1)

with torch.no_grad():
    u_pred = model(x)

# Analytical solution: u(x) = sin(pi*x) / pi^2
u_exact = torch.sin(np.pi * x) / (np.pi ** 2)

plot_solution_1d(
    x=x.numpy(),
    u_pred=u_pred.numpy(),
    u_true=u_exact.numpy(),
    title='Poisson Equation Solution'
)

In [ ]:
metrics = compute_all_metrics(u_pred.numpy(), u_exact.numpy())
print("\nError Metrics:")
for name, value in metrics.items():
    print(f"  {name}: {value:.6e}")